In [ ]:
%%capture
!pip install indiafactorlibrary

In [ ]:
# Notebook: 06_hypothesis_testing
# Series:   pyIndiaFactorInvesting
# Data:     Invespar Indian Factor Library + Nifty sample
# Run in:   Colab / Binder / local
# Author:   Rajan Raju

# 06 — Hypothesis Testing: From Intuition to Evidence

The previous notebooks built a toolkit: factor construction (02), regression mechanics (03), factor model application (03a), momentum deep dive (04), and fund decomposition (05). This notebook demonstrates how to use that toolkit to test a specific empirical claim about Indian equity markets — rigorously, honestly, and with full awareness of what can go wrong. The worked example walks through the complete hypothesis testing workflow: from a vague intuition to a precise claim, through operationalisation, testing, and robustness checks. The five pitfalls at the end are not hypothetical — they are the mistakes most commonly made in applied factor research.

**What you will leave with**
- A clear distinction between a vague intuition and a testable hypothesis
- A complete worked example: testing whether quality outperforms during market stress in India
- An understanding of how to operationalise variables, choose tests, and interpret results
- Five specific pitfalls that invalidate empirical claims, each illustrated with Indian data
- The skills to design and execute your own hypothesis test using the tools from this series

**Prerequisites:** All previous notebooks (00–05)

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
import statsmodels.formula.api as smf
import statsmodels.api as sm

sys.path.insert(0, str(Path('.').resolve().parent))

import indiafactorlibrary as ifl
from src.data import load_nifty_sample

In [ ]:
# ── ifl user-agent setup ─────────────────────────────────────────────────────
_lib = ifl.IndiaFactorLibrary()
_lib.session.headers.update({'User-Agent': (
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')})

def read(name):
    return _lib.read(name)

# ── FF6 full period ───────────────────────────────────────────────────────────
ff6_raw = read('FF6')[0];  ff6_raw.index = pd.to_datetime(ff6_raw.index)
ff6     = ff6_raw / 100
factor_cols = ['MF', 'SMB5', 'HML', 'RMW', 'CMA', 'WML']

# ── Nifty sample: aligned period ─────────────────────────────────────────────
nifty_raw = load_nifty_sample();  nifty_raw.index = pd.to_datetime(nifty_raw.index)
ff6_a     = ff6.reindex(nifty_raw.index).dropna()
nifty_a   = nifty_raw.reindex(ff6_a.index)
qual30_xs = nifty_a['nifty100_quality30'] - ff6_a['RF']

print(f"FF6 full period:  {ff6.index[0].strftime('%b %Y')} – {ff6.index[-1].strftime('%b %Y')}  ({len(ff6)} months)")
print(f"Aligned period:   {ff6_a.index[0].strftime('%b %Y')} – {ff6_a.index[-1].strftime('%b %Y')}  ({len(ff6_a)} months)")

## Section 1 — What Makes a Testable Hypothesis?

A hypothesis is not an opinion. "Quality stocks do well" is an opinion. "The monthly return of the RMW factor is positive and statistically significant in months where the market excess return (MF) is negative" is a testable hypothesis. The difference:

**A testable hypothesis must be:**
- **Falsifiable:** it must be possible for the data to contradict the claim. If no conceivable outcome would change your mind, you are not testing — you are confirming.
- **Specific:** the variables, the direction of the effect, and the conditions under which it is expected to hold must all be defined before looking at the data.
- **Operationally defined:** every term in the hypothesis must map to a measurable quantity. "Quality" must become RMW or a specific index. "Stress" must become a rule applied to observable data (e.g., MF < 0, or bottom quartile of MF).

**Common failure modes:**
- "Momentum works in India" — too vague. Which momentum measure? Over what period? Relative to what benchmark? "Works" how?
- "Small caps outperform" — outperform what? Over what horizon? Net of what costs?
- "This fund generates alpha" — alpha relative to what model? Over how many observations? At what significance level?

The worked example below takes a vague intuition from notebook 00 — quality indices appeared to outperform during the IL&FS crisis — and turns it into a formal test.

## Section 2 — Step 1: State the Claim Precisely

**The intuition (from notebook 00):** During the IL&FS crisis (mid-2018 to early 2019), the Nifty100 Quality 30 index held up better than the Nifty 100. This observation prompted the question: does quality systematically outperform during market stress in India?

**The vague claim:** "Quality outperforms during stress."

**The precise hypothesis:** "The mean monthly return of the RMW (Robust Minus Weak) factor is positive in months where the market excess return (MF) is negative, and this positive mean is statistically distinguishable from zero."

**Note what we have done:**
- "Quality" is operationalised as RMW — the Fama-French profitability factor from the Invespar Data Library
- "Outperforms" is operationalised as "positive mean return" — not just one good month, but a systematic tendency
- "Stress" is operationalised as "MF < 0" — months where the broad market delivered negative excess returns
- "Statistically distinguishable from zero" sets the evidentiary bar — we require not just a positive number but one that is unlikely to have arisen by chance

We will also test a second version using the Nifty100 Quality 30 index instead of RMW — because the distinction between a long-short factor and a long-only index matters (notebook 04 showed this for momentum). Note that the Quality30 comparison uses the shorter aligned period (Jan 2015 – Dec 2020).

In [ ]:
# ── Full period: RMW and stress definitions ───────────────────────────────────
rmw             = ff6['RMW']
mf              = ff6['MF']
stress_full     = mf < 0
non_stress_full = mf >= 0

# ── Aligned period: stress definition ────────────────────────────────────────
stress_a = ff6_a['MF'] < 0

n_stress = int(stress_full.sum())
n_total  = len(mf)
print(f"Full period: {n_total} months — {n_stress} stress ({n_stress/n_total:.1%}), "
      f"{n_total - n_stress} non-stress ({(n_total - n_stress)/n_total:.1%})")
print(f"Stress mean MF:     {mf[stress_full].mean()*100:+.2f}%")
print(f"Non-stress mean MF: {mf[non_stress_full].mean()*100:+.2f}%")
print(f"\nAligned period: {len(ff6_a)} months — {int(stress_a.sum())} stress, "
      f"{int((~stress_a).sum())} non-stress")

### 3b — Inspecting the Data Split

Before running any test, inspect the data split. Are there enough stress months for the test to have statistical power? Is the stress definition producing a meaningful separation in market conditions?

The output above shows the month counts and mean market returns in each regime. If the split is highly unbalanced (e.g., 5 stress months vs 200 non-stress months), the test will have low power regardless of the true effect. If mean MF in "stress" months is only modestly negative, the stress definition may not be capturing genuine market distress.

A useful benchmark: a one-sample t-test requires roughly 30+ observations for reliable inference under non-normality. With monthly data, stress periods that represent fewer than 20–25% of the sample may be insufficient for strong conclusions without additional robustness checks.

In [ ]:
# ── Alternative stress: bottom quartile of MF ────────────────────────────────
q25_full     = mf.quantile(0.25)
q25_a        = ff6_a['MF'].quantile(0.25)
stress_q25   = mf < q25_full
stress_q25_a = ff6_a['MF'] < q25_a

print("Stress definition comparison:")
print(f"  Full period — MF < 0 (original):     {int(stress_full.sum())} months")
print(f"  Full period — bottom quartile of MF: {int(stress_q25.sum())} months  "
      f"(threshold: {q25_full*100:.2f}%)")
print(f"\n  Aligned — MF < 0:                {int(stress_a.sum())} months")
print(f"  Aligned — bottom quartile of MF: {int(stress_q25_a.sum())} months  "
      f"(threshold: {q25_a*100:.2f}%)")
print("\nBoth definitions are used in Section 6 (robustness checks).")

## Section 4 — Step 3: Choose the Test

Three approaches are available, each with different strengths:

**Approach 1 — Conditional means with t-test:** Compute the mean RMW in stress months and in non-stress months separately. Test whether the stress-month mean is significantly different from zero using a one-sample t-test. Simple, transparent, easy to interpret. Weakness: does not control for other factors that may co-move with stress.

**Approach 2 — Sub-period regression:** Run the FF6 regression on stress months only and non-stress months only. Compare the RMW coefficient across the two sub-samples. This controls for other factor exposures but splits the sample, reducing statistical power.

**Approach 3 — Interaction term:** Run a single regression that includes an interaction between RMW and a stress dummy: `Y ~ MF + SMB5 + HML + RMW + CMA + WML + RMW:stress_dummy`. The coefficient on the interaction term measures how much the RMW effect changes during stress. Most powerful statistically (uses full sample) but harder to interpret.

We will use Approach 1 (transparent and directly answers the hypothesis) with Approach 3 as a robustness check.

In [ ]:
# ── Conditional means: RMW (full period) and Quality30 xs (aligned) ──────────
rmw_stress   = rmw[stress_full]
rmw_nostress = rmw[non_stress_full]
t_rmw, p_rmw = stats.ttest_1samp(rmw_stress, 0)

q30_stress   = qual30_xs[stress_a]
q30_nostress = qual30_xs[~stress_a]
t_q30, p_q30 = stats.ttest_1samp(q30_stress, 0)

res5a = pd.DataFrame({
    'Stress Mean (%)':     [rmw_stress.mean()*100,   q30_stress.mean()*100],
    'Non-Stress Mean (%)': [rmw_nostress.mean()*100, q30_nostress.mean()*100],
    't-stat':              [t_rmw,  t_q30],
    'p-value':             [p_rmw,  p_q30],
}, index=['RMW (full period)', 'Quality30 xs (aligned)'])
print(res5a.round(3).to_string())
print(f"\nRMW: {len(rmw_stress)} stress months | Quality30: {len(q30_stress)} stress months")

### 5b — Reading the Conditional Means Table

Read the table. For the hypothesis to be supported, the stress-month mean of RMW must be positive and the p-value must be below the chosen significance level (conventionally 0.05, though Harvey, Liu and Zhu 2016 recommend |t| > 3 for factor research given the multiple-testing environment in empirical finance).

If the mean is positive but the p-value is above 0.05, the data are consistent with the hypothesis but also consistent with zero — we cannot distinguish signal from noise with this sample.

Compare RMW (long-short academic factor) to Quality30 excess return (long-only index). If Quality30 shows different behaviour from RMW during stress, this echoes the long-only vs long-short distinction from notebook 04: the investable product and the academic factor can tell different stories. Note that the two series also cover different periods, so period effects may contribute to any difference.

In [ ]:
# ── Interaction regression: Quality30 xs on aligned period ───────────────────
df5c = ff6_a[factor_cols].copy()
df5c['qual30_xs']  = qual30_xs.values
df5c['stress_d']   = stress_a.astype(int).values
df5c['RMW_stress'] = df5c['RMW'] * df5c['stress_d']

formula = 'qual30_xs ~ MF + SMB5 + HML + RMW + CMA + WML + RMW_stress'
mod5c = smf.ols(formula, data=df5c).fit(
    cov_type='HAC', cov_kwds={'maxlags': 3})
print(mod5c.summary())

### 5d — Interpreting the Interaction Term

The interaction term coefficient (`RMW_stress`) tells you how the RMW effect changes during stress months. A positive, significant coefficient means RMW contributes more to Quality30 returns during stress than during non-stress — supporting the hypothesis. A non-significant interaction means the quality effect does not differ between regimes in a statistically detectable way.

Interpret honestly: a null result is not a failure. It means the data, over this sample, do not provide evidence for the claim. The claim may still be true — the sample may be too short, the stress definition too crude, or the effect too small to detect with monthly data. Labelling a null result as "inconclusive" rather than "false" is the correct scientific stance.

## Section 6 — Step 5: Check Robustness

A single test is never enough. Robustness checks ask: does the result survive when we change the assumptions?

**Three robustness dimensions:**
- **Alternative stress definition:** use bottom quartile of MF instead of MF < 0 (more restrictive, captures the worst 25% of market months)
- **Alternative quality measure:** use CMA (investment conservatism) instead of RMW — a different empirical dimension of "quality"
- **Alternative period:** split the aligned sample in half and test each sub-period separately

If the result survives all three, it is more credible. If it survives none, the original finding was likely fragile or sample-specific. Mixed results — where the finding holds under some changes but not others — are the norm in empirical finance, and honest reporting requires showing them all.

In [ ]:
# ── Robustness 1: bottom-quartile stress, RMW, full period ───────────────────
t_rmw_q25, p_rmw_q25 = stats.ttest_1samp(rmw[stress_q25], 0)
# ── Robustness 2: CMA instead of RMW, full period ────────────────────────────
cma_full = ff6['CMA']
t_cma, p_cma = stats.ttest_1samp(cma_full[stress_full], 0)
# ── Robustness 3: sub-period split, Quality30, aligned ───────────────────────
mid = len(qual30_xs) // 2
q30_h1, stress_h1 = qual30_xs.iloc[:mid], stress_a.iloc[:mid]
q30_h2, stress_h2 = qual30_xs.iloc[mid:], stress_a.iloc[mid:]
t_h1, p_h1 = stats.ttest_1samp(q30_h1[stress_h1], 0)
t_h2, p_h2 = stats.ttest_1samp(q30_h2[stress_h2], 0)

rob = pd.DataFrame({
    'Stress Mean (%)': [rmw_stress.mean()*100, rmw[stress_q25].mean()*100,
                        cma_full[stress_full].mean()*100,
                        q30_h1[stress_h1].mean()*100, q30_h2[stress_h2].mean()*100],
    't-stat':  [t_rmw, t_rmw_q25, t_cma, t_h1, t_h2],
    'p-value': [p_rmw, p_rmw_q25, p_cma, p_h1, p_h2],
}, index=['RMW — original (MF<0)', 'RMW — alt stress (Q25)',
          'CMA — alt quality (MF<0)', 'Quality30 — first half', 'Quality30 — second half'])
print(rob.round(3).to_string())

### 6c — Reading the Robustness Table

The robustness table shows whether the original finding holds under different specifications. Look for consistency: if the stress-month mean is positive across all rows, the finding is robust to the changes tested here. If it is positive in some rows but negative in others, the finding is sensitive to the specification choice.

Describe what the table shows — do not assert a conclusion before looking at the output. Mixed results are the norm in empirical finance. A finding that only appears under one stress definition, one quality measure, and one sub-period is less credible than one that appears consistently.

Harvey, Liu and Zhu (2016) note that a t-statistic of 3.0 (p ≈ 0.003) is a more appropriate bar for factor research than the conventional 1.96 (p = 0.05), precisely because researchers test many specifications before reporting results.

## Section 7 — What Can Go Wrong: Five Pitfalls

### Pitfall 1 — Data Snooping

You looked at the data, noticed quality outperformed during IL&FS, and formed the hypothesis. This is data snooping: the hypothesis was suggested by the same data used to test it. The probability of finding *some* pattern in any dataset is close to 1 — the question is whether the pattern reflects a real economic mechanism or a coincidence.

**The honest approach:** acknowledge the data snooping explicitly (as done in Section 2), set a higher significance bar (Harvey, Liu and Zhu 2016 recommend |t| > 3 for factor research), and check out-of-sample if possible. In this case, out-of-sample would mean testing the hypothesis on a different stress episode not used to motivate the claim — for example, the COVID crash of February–March 2020, or the GFC period of 2008–2009.

Data snooping does not invalidate the finding — it means the threshold for credibility is higher and out-of-sample confirmation is more important.

### Pitfall 2 — Multiple Testing

If you test 20 hypotheses at the 5% significance level, you expect one to be significant by chance alone. The more hypotheses tested — different factors, different stress definitions, different periods — the more likely a false positive. This is why the robustness section in Section 6 reports all tests conducted, not just the one that "worked."

**Bonferroni correction** (divide the significance level by the number of tests) is the simplest fix. With five tests in Section 6 (one original plus four robustness checks), the Bonferroni-corrected threshold is 0.05 / 5 = 0.01. This is conservative — all five tests are correlated — but it makes the bar explicit.

The key discipline is to precommit to the tests before looking at results, report all of them, and not search for the specification that gives a significant result.

### Pitfall 3 — Survivorship Bias

The Nifty100 Quality 30 index exists today because quality stocks performed well. Indices that performed poorly were never created or were quietly discontinued. Testing the hypothesis on a surviving index biases toward finding a positive result — only the successful expressions of the quality idea made it to market.

The RMW factor, constructed from the full universe including dead and delisted firms, is less susceptible to this bias — but not immune. Factor construction methods that look backward to identify the "quality" characteristic can still embed snooping bias if the characteristic itself was selected post-hoc.

In practice, the Invespar factor library follows the Fama-French methodology with a fixed construction rule, which reduces (but does not eliminate) this concern. Custom factor analysis should always verify that the construction rule was fixed before the test period begins.

### Pitfall 4 — Look-Ahead Bias

The IL&FS crisis unfolded from September 2018. If your "quality" measure uses balance sheet data from September 2018 filings to define quality *in* September 2018, you are using information that was not available to investors at the time. The Invespar factor library avoids this by using a six-month lag — March fiscal year data applied to September portfolio formation — but custom analyses must check timing carefully.

Common sources of look-ahead bias:
- Using end-of-month prices to define portfolio weights at the start of the month
- Using annual report data before the report was published
- Winsorising or normalising data using statistics from the full sample rather than an expanding in-sample window

The test in this notebook is not subject to look-ahead bias because it uses the Invespar factors directly. Any extension that introduces custom data needs to verify the timing of information availability.

### Pitfall 5 — Confusing Statistical and Economic Significance

A statistically significant result (p < 0.05) tells you the effect is unlikely to be zero. It does not tell you the effect is large enough to matter. If RMW averages +15 basis points per month during stress, that is about 1.8% per year — meaningful for some contexts, irrelevant for others.

Always report the magnitude alongside the significance, and consider whether transaction costs, taxes, and implementation frictions would consume the effect. Notebook 04 documented total implementation costs of approximately 3.7% per annum for a momentum winner portfolio (Raju and Teli 2022, SSRN 4000418). A quality tilt with similar turnover could face similar costs.

Conversely, a non-significant result with a large estimated effect and a small sample is not evidence of no effect — it is evidence of insufficient power. The distinction matters: "the data do not support the hypothesis" is different from "the hypothesis is false."

## Section 8 — Series Wrap-Up

### 8a — What You Can Now Do

Over eight notebooks, this series has built a complete toolkit for factor-based analysis of Indian equities:

| Notebook | Capability |
|----------|------------|
| 00 — Indian Market Puzzles | Observe anomalies in Indian data that motivate factor research |
| 01 — Factor Zoo | Understand which factors have empirical support and which do not |
| 02 — Factor Construction | Read and use the Invespar Data Library; understand what is inside the sub-portfolios |
| 03 — Intro to Asset Pricing | Run and interpret regressions with appropriate standard errors; distinguish factors from style indices |
| 03a — Factor Models | Apply FF6 systematically to indices; detect time-varying exposures with rolling regressions |
| 04 — Momentum Deep Dive | Analyse a single factor in depth: crashes, interactions, implementation frictions |
| 05 — Fund Decomposition | Decompose fund returns into factor exposures, three alpha measures, and RBSA replication |
| 06 — Hypothesis Testing | Test a specific claim with proper methodology and awareness of pitfalls |

The Invespar Indian Factor Library (https://invespar.com/research/) provides regularly updated factor data for Indian equities. The `indiafactorlibrary` Python package makes this data accessible in a single line of code. Both are freely available.

### 8b — Where to Go Next

This series covers the analytical foundations. Natural extensions include:

- **Factor timing and regime analysis:** when do factors work, and can regimes be identified in advance?
- **Multi-factor portfolio construction:** how to combine factors with different risk profiles and correlations?
- **Transaction cost modelling:** what survives implementation, accounting for brokerage, market impact, and taxes?
- **Extended universe analysis:** how do factor exposures and premia change beyond the top 200 or 500 stocks?

These topics require additional data, more sophisticated models, and careful attention to the gap between theory and implementation — the same gap this series has emphasised throughout.

**What we established in this notebook**

| Section | Finding |
|---------|--------|
| 1 | Testable hypotheses require falsifiability, specificity, and operational definition |
| 2–3 | The quality-stress hypothesis operationalises RMW, stress months, and the significance threshold |
| 4–5 | Conditional means and interaction regressions provide complementary tests |
| 6 | Robustness across three dimensions is the minimum credibility standard |
| 7 | Data snooping, multiple testing, survivorship, look-ahead, and magnitude are the five key pitfalls |

## References

- Daniel, K. and Moskowitz, T.J. (2016). Momentum Crashes. *Journal of Financial Economics*, 122(2), 221–247.
- Harvey, C.R., Liu, Y. and Zhu, H. (2016). ...and the Cross-Section of Expected Returns. *Review of Financial Studies*, 29(1), 5–68.
- Jegadeesh, N. and Titman, S. (1993). Returns to Buying Winners and Selling Losers: Implications for Stock Market Efficiency. *Journal of Finance*, 48(1), 65–91.
- Raju, R. (2022). Factor Exposures and Alpha of Indian Equity Schemes. SSRN Working Paper 4107950.
- Raju, R. (2022). Four and Five-Factor Models in the Indian Equities Market. SSRN Working Paper 4054146.
- Raju, R. and Teli, A. (2022). "Long" Factors, not "Short" Change: Long Only Factor Portfolios in India. SSRN Working Paper 4000418.
- Invespar Indian Factor Library. https://invespar.com/research/